In [ ]:
#installs
!pip install --upgrade --quiet  langchain_google_community
!pip install huggingface_hub
!pip install --upgrade langchain
!pip install langchain-community langchain-core
!pip install -U langchain-huggingface
!pip install --upgrade gradio

GOOGLE CUSTOM SEARCH TOOL

In [ ]:
#IMPORTS
import os
from langchain_core.tools import Tool
from langchain_google_community import GoogleSearchAPIWrapper
from google.colab import userdata

In [ ]:
#ENVIRONMENT SETUP
os.environ["GOOGLE_CSE_ID"] = userdata.get('GOOGLE_CSE_ID')
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

#CREATING GOOGLE SEARCH TOOL
search = GoogleSearchAPIWrapper()

tool = Tool(
    name="google_search",
    description="Search Google for recent results.",
    func=search.run,
)

MODEL

In [ ]:
#IMPORTS
import torch
from huggingface_hub import login
from langchain_community.llms import HuggingFaceEndpoint
from langchain_community.chat_models.huggingface import ChatHuggingFace

In [ ]:
#LOGIN
login(token=userdata.get('HGtoken'))

#CREATING MODEL WITH ENDPOINT
llm=HuggingFaceEndpoint(repo_id="HuggingFaceH4/zephyr-7b-beta")
local_llm=ChatHuggingFace(llm=llm)

EMBEDDING AGENT

In [ ]:
#IMPORTS
from langchain.agents import initialize_agent, load_tools, AgentExecutor, create_structured_chat_agent
from langchain import hub
from langchain.memory import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain import PromptTemplate

In [ ]:
#PROMPT FOR AGENT
prompt = hub.pull("hwchase17/structured-chat-agent")


#CHAT MEMORY STORAGE
memory = ChatMessageHistory(session_id="session")

#AGENT
agent=create_structured_chat_agent(local_llm,
                                   [tool],
                                   prompt
                                   )
#AGENT_EXECUTOR
agent_executor = AgentExecutor(agent=agent, tools=[tool],
                               handle_parsing_errors=True,
                               max_iterations=10
                               )

#EXECUTABLE AGENT WITH MEMORY
agent_with_chat_history = RunnableWithMessageHistory(agent_executor,
                                                     lambda session_id: memory,
                                                     input_messages_key="input",
                                                     history_messages_key="chat_history"
                                                    )

USER INTERFACE

In [ ]:
#imports
import gradio as gr

In [ ]:
#Chatbot conversation function
def chatbot_response_conversation(message, history):
  answer = agent_with_chat_history.invoke({"input":message},
                                          config={"configurable": {"session_id": "<foo>"}})
  return answer['output']

In [ ]:
#chatbot UI
travel_agent = chatbot_conversation_ui = gr.ChatInterface(chatbot_response_conversation,
                              title="Travel Agent")

In [ ]:
# Launch the app
travel_agent.launch(inbrowser=True, share=True)